# Notebook 3: ML Model Training
Goal: Train XGBoost to predict missing freight rates. Evaluate with RMSE, R², MAPE. Explain with SHAP.

In [ ]:
import sys
import os
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROCESSED_DIR = '../data/processed/'
MODELS_DIR    = '../models/'
print('Environment ready.')

## 1. Train/Val/Test Split

In [ ]:
from src.data.feature_pipeline import load_features

df = load_features()  # loads data/processed/features_long.parquet

# Only rows with observed freight_rate are used for supervised learning
obs = df[df['freight_rate'].notna()].copy()
obs['year'] = obs['year'].astype(int)

print('Year distribution of observed rows:')
print(obs['year'].value_counts().sort_index().to_string())

# Temporal split: train ≤ 2018, val = 2019, test ≥ 2020
train = obs[obs['year'] <= 2018]
val   = obs[obs['year'] == 2019]
test  = obs[obs['year'] >= 2020]

print(f'\nTrain rows: {len(train):,} (years ≤ 2018)')
print(f'Val   rows: {len(val):,}  (year 2019)')
print(f'Test  rows: {len(test):,}  (years ≥ 2020)')
print(f'Total observed: {len(obs):,}')

## 2. Train XGBoost

In [ ]:
from src.models.train_xgb import train_and_evaluate

# train_and_evaluate() performs:
#   1. Loads features_long.parquet
#   2. Temporal train/val/test split
#   3. Fits XGBRegressor with early stopping on val set
#   4. Predicts on test set; saves predictions to models/test_predictions.parquet
#   5. Fits SHAP TreeExplainer; saves explainer to models/shap_explainer.pkl
#   6. Runs model.predict() on ALL rows to impute missing freight_rates
#   7. Saves graph_edges_full.parquet to data/processed/

model, explainer = train_and_evaluate()
print('Training complete.')
print(f'Model type: {type(model).__name__}')
print(f'Model params: n_estimators={model.n_estimators}, max_depth={model.max_depth}, lr={model.learning_rate}')

## 3. Evaluation Results

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

preds = pd.read_parquet(MODELS_DIR + 'test_predictions.parquet')
y_true = preds['freight_rate']
y_pred = preds['predicted_freight_rate']

rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
r2    = r2_score(y_true, y_pred)
mape  = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
mae   = np.mean(np.abs(y_true - y_pred))

metrics = pd.DataFrame({
    'Metric': ['RMSE (USD/TEU)', 'MAE (USD/TEU)', 'R²', 'MAPE (%)'],
    'Value':  [f'{rmse:,.1f}', f'{mae:,.1f}', f'{r2:.4f}', f'{mape:.2f}']
})
print('Test Set Evaluation Metrics:')
display(metrics)

# Scatter plot: predicted vs actual
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_true, y_pred, alpha=0.3, s=10, color='steelblue', label='Predictions')
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
ax.set_xlabel('Actual Freight Rate (USD/TEU)')
ax.set_ylabel('Predicted Freight Rate (USD/TEU)')
ax.set_title(f'Predicted vs Actual — Test Set\n(R²={r2:.4f}, RMSE={rmse:,.0f})')
ax.legend()
plt.tight_layout()
plt.show()

## 4. COVID Direction Check

In [ ]:
# Compare mean predicted freight rates for pre-COVID vs COVID years
check_years = [2018, 2019, 2020, 2021]
pred_all = pd.read_parquet(MODELS_DIR + 'test_predictions.parquet')

# Load full features to include 2018/2019 (which may be in val/train sets)
df_full = load_features()
df_full['year'] = df_full['year'].astype(int)

ML_FEATURES = [
    'bilateral_lsci', 'origin_lsci', 'dest_lsci',
    'origin_teu', 'dest_teu', 'origin_fleet_dwt', 'dest_fleet_dwt',
    'origin_port_calls', 'dest_port_calls',
    'trade_value_usd', 'year_numeric', 'is_covid',
    'freight_rate_lag1', 'freight_rate_delta1',
]
avail = [f for f in ML_FEATURES if f in df_full.columns]

year_means = {}
for yr in check_years:
    subset = df_full[df_full['year'] == yr][avail].fillna(0)
    if len(subset) > 0:
        preds_yr = model.predict(subset)
        year_means[yr] = float(np.mean(preds_yr))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([str(y) for y in year_means], list(year_means.values()),
              color=['steelblue', 'steelblue', 'coral', 'coral'])
ax.set_ylabel('Mean Predicted Freight Rate (USD/TEU)')
ax.set_title('COVID Direction Check: Mean Predicted Rates by Year')
for bar, val in zip(bars, year_means.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

if 2018 in year_means and 2020 in year_means:
    pct_increase = (year_means[2020] - year_means[2018]) / year_means[2018] * 100
    print(f'\nPredicted rate increase from 2018 → 2020: +{pct_increase:.1f}%')
    assert year_means[2020] > year_means[2018], 'FAIL: Expected 2020 > 2018'
    print('Direction check PASSED: Model correctly predicts upward shift in 2020.')

## 5. SHAP Analysis

In [ ]:
import shap
import pickle

# Load SHAP explainer saved during training
with open(MODELS_DIR + 'shap_explainer.pkl', 'rb') as f:
    explainer = pickle.load(f)

# Compute SHAP values on a sample of the test set
test_sample = df_full[df_full['year'] >= 2020][avail].fillna(0).head(500)
shap_values = explainer(test_sample)

# Mean absolute SHAP per feature
mean_abs_shap = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=test_sample.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
mean_abs_shap.plot.barh(ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Mean |SHAP value| (impact on freight rate prediction)')
ax.set_title('SHAP Feature Importance — Test Set Sample (n=500)')
plt.tight_layout()
plt.show()

print('Top 5 most impactful features:')
print(mean_abs_shap.sort_values(ascending=False).head(5).to_string())

## 6. SHAP Waterfall Example

In [ ]:
# Show SHAP waterfall for China → USA, product 8517, year 2021
mask = (
    (df_full['origin'].str.lower().str.contains('china')) &
    (df_full['dest'].str.lower().str.contains('united states|usa')) &
    (df_full['product_code'].astype(str) == '8517') &
    (df_full['year'].astype(int) == 2021)
)
example_rows = df_full[mask][avail].fillna(0)

if len(example_rows) == 0:
    print('Exact row not found — using closest proxy (first 2021 row).')
    example_rows = df_full[df_full['year'].astype(int) == 2021][avail].fillna(0).head(1)

shap_example = explainer(example_rows.iloc[[0]])
shap.waterfall_plot(shap_example[0], max_display=12, show=False)
plt.title('SHAP Waterfall: China → USA, HS 8517, 2021')
plt.tight_layout()
plt.show()

predicted_val = float(model.predict(example_rows.iloc[[0]])[0])
print(f'Predicted freight rate for this corridor: USD {predicted_val:,.0f} / TEU')

## 7. Imputation Results

In [ ]:
edges = pd.read_parquet(PROCESSED_DIR + 'graph_edges_full.parquet')

print(f'Total graph edges: {len(edges):,}')
print(f'Columns: {edges.columns.tolist()}')

# Imputation stats
n_observed  = edges['is_observed'].sum()
n_predicted = (~edges['is_observed']).sum()
pct_pred    = n_predicted / len(edges) * 100
print(f'\nObserved edges:  {n_observed:,}  ({100 - pct_pred:.1f}%)')
print(f'Predicted edges: {n_predicted:,}  ({pct_pred:.1f}%)')

# Side-by-side distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

obs_rates  = edges[edges['is_observed']]['freight_rate'].dropna()
pred_rates = edges[~edges['is_observed']]['freight_rate'].dropna()

axes[0].hist(obs_rates,  bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title(f'Observed Rates\n(n={len(obs_rates):,})')
axes[0].set_xlabel('Freight Rate (USD/TEU)')

axes[1].hist(pred_rates, bins=50, color='coral',     edgecolor='white', alpha=0.85)
axes[1].set_title(f'XGBoost-Imputed Rates\n(n={len(pred_rates):,})')
axes[1].set_xlabel('Freight Rate (USD/TEU)')

for ax in axes:
    ax.set_ylabel('Count')

fig.suptitle('Distribution of Freight Rates: Observed vs Imputed', fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nObserved  — median: {obs_rates.median():,.0f}, mean: {obs_rates.mean():,.0f}')
print(f'Imputed   — median: {pred_rates.median():,.0f}, mean: {pred_rates.mean():,.0f}')